In [28]:
import numpy as np
import pandas as pd
from astropy.io import fits

from astropy.time import Time, TimeDelta
import astropy.units as u

import random

In [29]:
'Make sure to get the csv file'

df = pd.read_csv("exposures-daily.csv")
program_b = ['bright', 'bright1b'] # these are the programs you can use to avoid manually typing the list all the time LOL
program_d = ['dark', 'dark1b'] 

"""
For extracting data. You can either extract data from a range of nights or a single night 
    (_range = True will give back data over a range ; _range = False will give data from the nights you provide
All data must be before January 1, 2026 (> 20260101) 
Honestly, you probably will only need a single date for timestamps, but in case you need a range or something, you have 
options!

Note: 
the input for the parameter "date" is in YYYYMMDD (ex. 20260821)
"""
def data_extraction(date, program=None, _range=False):
    mask = (df['SURVEY'] == 'main') & (~df['PROGRAM'].isin(['other', 'backup'])) & (df['EFFTIME_SPEC'] > 0)
    if (_range == True): 
        if type(date) is list:
            mask &= (df['NIGHT'].between(date[0], date[1])) 
            # Returns data from the range of dates given 
        elif type(date) is int:
            mask &= (df['NIGHT'].between(20260101, date))
            # Returns data from the range from Jan 1, 2026 to the date given
    elif (_range == False): 
        if type(date) is list:
            mask &= (df['NIGHT'].isin(date))
            # Returns data from the each night given 
        elif type(date) is int: 
             mask &= (df['NIGHT'] == date)
            # Returns data from night given 
    _data = df[mask]
    
    # Masking the data for a single type of program: bright vs. dark 
    data = _data[_data['PROGRAM'].isin(list(program))]
    tile, exptime = data['TILEID'].unique(), (data.groupby('TILEID')['EXPTIME'].sum()).values 
        # Exposure times will be added to compensate multiple tiles being observed across different times 

    return tile, exptime, data # Returns the tile IDs and exposure time per tile for each night given 

In [30]:
"""
Percentile Ranges:
    BRIGHT program: [10,85]
    DARK programs: [10,78]

P.S.: Use one day since most days have a couple of times (some have multiple timestamps as well). Also, some days might not 
have a single bright/dark tile at all
NOTE (mostly reworded from Dylan): 
    1. The timestamp is when the tile STARTS
    2. You will notice that the timestamp given will not match the date that you input in. This is because the 'NIGHT' column 
    is to the current time at the TELESCOPE in mountain time (~ 7 hours behind UTC) and is set to the day the observing night 
    starts (i.e. NIGHT = 20260409 is the night of 4/9/2026 to morning of 4/10/2026). And it seems that a majority of the 
    timestamps are given on the morning of the next day 

"""

def tiles_list(date, pc_range, program, _range=False):
    t, expt, data = data_extraction(date, program, _range)
    if len(t)==0:
        return("No tiles were available for the nights given")
    low, high = np.percentile(expt, pc_range)    
    good_tiles = t[(expt >= low) & (expt <= high)]
    bad_tiles = t[(expt < low) | (expt > high)]
        
    timestamps = []
    def conversion(mjd):
        return Time(mjd, format = 'mjd', scale = 'utc').isot
    for tile in list(good_tiles): 
        mjd_times = data.loc[(df['TILEID'] == tile), 'MJD']
        mjd_times_list = []
        for mjd in mjd_times: 
            mjd_times_list.append(conversion(mjd))
        timestamps.append(mjd_times_list)
                
    return f"    Good Tiles: {good_tiles}\n\n    Timestamps for good tiles: {timestamps}\n\n\n    Bad Tiles: {bad_tiles}"

In [31]:
date = 20260409 #example date
print(f"BRIGHT TILES:\n{tiles_list(date, [10,85], program_b)}\n\n\n")
print(f"DARK TILES:\n{tiles_list(date, [10,78], program_d)}")

BRIGHT TILES:
    Good Tiles: [120423 120455 121157]

    Timestamps for good tiles: [['2026-04-10T02:51:37.321'], ['2026-04-10T03:10:15.439', '2026-04-10T03:28:20.942'], ['2026-04-10T12:00:42.276']]


    Bad Tiles: [120136 120118]



DARK TILES:
    Good Tiles: [100807]

    Timestamps for good tiles: [['2026-04-10T11:42:06.662']]


    Bad Tiles: []


In [32]:
random_date = (df.loc[(df['NIGHT'] > 20260101, 'NIGHT')].sample()).item()
"""
if you just want random dates, you can use this random date generator above
"""
print(random_date)
print(f"BRIGHT TILES:\n{tiles_list(random_date, [10,85], program_b)}\n\n\n")
print(f"DARK TILES:\n{tiles_list(random_date, [10,78], program_d)}")

20261028
BRIGHT TILES:
    Good Tiles: [23684]

    Timestamps for good tiles: [['2026-10-29T05:09:40.121', '2026-10-29T05:35:40.611']]


    Bad Tiles: [123246  32046]



DARK TILES:
    Good Tiles: [101191 101187 101175]

    Timestamps for good tiles: [['2026-10-29T02:03:19.178'], ['2026-10-29T03:39:57.146'], ['2026-10-29T04:14:02.748', '2026-10-29T04:36:53.999']]


    Bad Tiles: [102626 100263 101188]
